# 01 · The Chat Multi-Agent Graph, End to End

This notebook builds and runs the **chat multi-agent graph** at the heart of
SkyLab Manufacturing Intelligence: a **Supervisor + 3 Specialist** pattern on
[Strands Agents](https://github.com/strands-agents/sdk-python).

We import the *actual* agent factories from `agent/agents.py` — the same code
the FastAPI backend and the deployed AgentCore Runtime use — and drive them by
hand so you can watch each stage of the graph:

```
PLANNER (Haiku 5.5)  ->  ordered specialist plan   (no tools, pure routing)
    |
    +--> CATALOG     (Sonnet 5.5)  discovery + KB retrieval
    +--> EVALUATION  (Sonnet 5.5)  deterministic spec-fit + visual similarity
    +--> KNOWLEDGE   (Sonnet 5.5)  datasheet/ICD facts + live web/EOL
    |
    v
SUPERVISOR (Opus 5.5)  ->  synthesis  (ONLY when 2+ specialists ran)
```

**The design idea:** routing is a cheap classification task (Haiku), each
specialist is narrow and tool-focused (Sonnet), and the expensive synthesis
model (Opus) is reserved for the one job that actually needs it — merging
multiple specialists into a single coherent answer.

> These agents assemble evidence. They never recommend or approve a part
> substitution — the engineer decides.

## 0 · Setup

Run Jupyter from the **repo root** so `import agent...` resolves. The cell
below adds the repo root to `sys.path` and selects the dataset profile
(`synthetic` = the AnyCompany reaction-wheel catalog that ships in the repo).

You need AWS credentials with Bedrock access in `us-east-1` and model access
for Claude Haiku / Sonnet / Opus 5.5.

In [1]:
import os, sys, json
from pathlib import Path

# Make the repo root importable whether Jupyter was launched from the root
# or from inside notebooks/.
# Standalone: use the vendored mei_core package + bundled data in this folder.
# No dependency on the parent app's agent/ or api/ — this folder is portable.
NB_DIR = Path.cwd() if (Path.cwd() / 'mei_core').exists() else Path.cwd() / 'notebooks'
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))
REPO_ROOT = NB_DIR  # alias: bundled demo-crops/ and evals/ live under NB_DIR
_bundled = NB_DIR / 'data' / 'synthetic-parts-data'
if _bundled.exists():
    os.environ['PARTS_DATA_DIR'] = str(_bundled)

# Select the catalog + region BEFORE importing agent code (tools.py reads
# these at import time via dataset_profiles.resolve()).
os.environ.setdefault('DATASET', 'synthetic')
os.environ.setdefault('BEDROCK_REGION', 'us-east-1')

from mei_core import dataset_profiles
print(json.dumps(dataset_profiles.resolve(), indent=2))

{
  "DATASET": "synthetic",
  "PARTS_DATA_DIR": "/Users/agavili/Desktop/SkyLab-ManufacturingIntelligence/notebooks/data/synthetic-parts-data",
  "PARTS_KB_ID": "KBJSEK4HDV",
  "S3_VECTOR_BUCKET": "skylab-mfg-vectors",
  "S3_VECTOR_INDEX": "reaction-wheels-synthetic"
}


## 1 · Meet the agents

Each agent is created by a factory function in `agent/agents.py`. The factory
sets the model, the system prompt, the toolset, and an explicit output-token
ceiling. Notice the **model per role** and the **tools per specialist** — this
is where the narrow-specialist design lives.

In [2]:
from mei_core.agents import (
    create_planner,
    create_catalog_specialist,
    create_evaluation_specialist,
    create_knowledge_specialist,
    create_supervisor,
    MODEL_ID, SPECIALIST_MODEL_ID, PLANNER_MODEL_ID,
)

print('planner    :', PLANNER_MODEL_ID)
print('specialists:', SPECIALIST_MODEL_ID)
print('supervisor :', MODEL_ID)

planner    = create_planner()
catalog    = create_catalog_specialist()
evaluation = create_evaluation_specialist()
knowledge  = create_knowledge_specialist()
supervisor = create_supervisor()

specialists = {'catalog': catalog, 'evaluation': evaluation, 'knowledge': knowledge}

for name, agent in specialists.items():
    tool_names = [getattr(t, '__name__', getattr(t, 'tool_name', str(t)))
                  for t in getattr(agent, 'tool_names', []) or []]
    # Strands exposes registered tools a few different ways across versions;
    # fall back to the tool spec if tool_names isn't populated.
    if not tool_names:
        tool_names = list(getattr(agent, 'tool_config', {}).get('tools', []))
    print(f'{name:10s} -> {len(tool_names) or "?"} tools')

planner    : us.anthropic.claude-haiku-5-5
specialists: us.anthropic.claude-sonnet-5-5
supervisor : us.anthropic.claude-opus-5-5


catalog    -> 4 tools
evaluation -> 5 tools
knowledge  -> 3 tools


## 2 · The planner: routing as classification

The planner's **only** job is to emit an ordered list of specialists. It has no
tools and uses the fast Haiku model. It is deliberately given a tight,
single-purpose system prompt (`PLANNER_SYSTEM_PROMPT`) so it classifies rather
than writing prose.

The exact per-call prompt below is copied verbatim from `run_multi_agent()` in
`agent/agents.py` — we reproduce it here so the routing you see matches the app.

In [3]:
from mei_core.agents import _parse_specialist_plan

def plan_route(user_input: str):
    planning_prompt = (
        f'An engineer asks: "{user_input}"\n\n'
        'Plan which specialists to use and in what ORDER. Complex queries may need '
        'multiple specialists in sequence (e.g., Knowledge first to check status, '
        'then Catalog to find alternatives, then Evaluation to score them). Most '
        'queries only need ONE specialist — only chain multiple when the query '
        'genuinely requires separate steps.\n\n'
        'Return ONLY a numbered list, nothing else. No explanation, no preamble, '
        'no markdown formatting — just plain numbered lines, in this exact format:\n'
        '1. KNOWLEDGE — check EOL status via web search\n'
        '2. CATALOG — find alternatives matching requirements\n'
        '3. EVALUATION — score candidates against requirements\n\n'
        'For simple queries, a single specialist is fine:\n'
        '1. CATALOG — discover parts matching the description'
    )
    plan_text = str(planner(planning_prompt))
    ordered = _parse_specialist_plan(plan_text)
    return plan_text, ordered

plan_text, ordered = plan_route('Which reaction wheels have at least 0.05 Nms of momentum storage?')
print('--- planner raw output ---')
print(plan_text)
print('--- parsed routing plan ---')
print(ordered)

1. CATALOG — find reaction wheels with momentum storage of at least 0.05 Nms--- planner raw output ---
1. CATALOG — find reaction wheels with momentum storage of at least 0.05 Nms

--- parsed routing plan ---
['catalog']


**What just happened.** The planner saw a *discovery* question and routed to a
single specialist: `['catalog']`. `_parse_specialist_plan` extracts the ordered
specialist names from the numbered list and ignores any stray prose. If the
planner emits nothing parseable, it defaults to `['catalog']` so the graph never
stalls.

Try a few routing calls to see the taxonomy in action:

In [4]:
examples = [
    'Compare the RW3 0.06 against the RW 0.03 for a 0.05 Nms slot',
    'What is the flight heritage of the RW3 0.06?',
    'Is the RW 0.03 going end-of-life? If so, find alternatives and score them.',
]
for q in examples:
    _, route = plan_route(q)
    print(f'{route}  <-  {q}')

1. EVALUATION — compare RW3 0.06 against RW 0.03 for the 0.05 Nms requirement['evaluation']  <-  Compare the RW3 0.06 against the RW 0.03 for a 0.05 Nms slot


1. KNOWLEDGE — retrieve flight heritage for the RW3 0.06['knowledge']  <-  What is the flight heritage of the RW3 0.06?


1. KNOWLEDGE — check EOL status of the RW 0.03 via web search
2. CATALOG — find alternatives matching the RW 0.03 requirements
3. EVALUATION — score candidates against requirements['knowledge', 'catalog', 'evaluation']  <-  Is the RW 0.03 going end-of-life? If so, find alternatives and score them.


Notice the last query chains **three** specialists
(`['knowledge', 'catalog', 'evaluation']`) — a status check feeds a discovery
step, which feeds a scoring step. That sequential hand-off is the whole reason a
planner exists instead of a fixed route.

## 3 · Run a single-specialist query

When the plan has exactly one specialist, that specialist's answer **is** the
final answer — the supervisor synthesis step is skipped (a second Opus call just
to reword one complete answer is pure latency). Let's run the catalog discovery
query end to end.

In [5]:
def run_graph(user_input: str, verbose: bool = True) -> dict:
    """A compact, readable re-implementation of run_multi_agent() from
    agent/agents.py, with print statements so you can watch each stage."""
    plan_text, ordered = plan_route(user_input)
    if verbose:
        print(f'ROUTE: {ordered}\n')

    steps, accumulated = [], ''
    for spec_name in ordered:
        specialist = specialists[spec_name]
        if accumulated:
            prompt = (
                f'Engineer\'s original question: "{user_input}"\n\n'
                f'Context from prior specialists:\n{accumulated}\n\n'
                'Now do your part. Use your tools to add your contribution.'
            )
        else:
            prompt = user_input
        if verbose:
            print(f'--- {spec_name.upper()} specialist running ---')
        text = str(specialist(prompt))
        steps.append({'specialist': spec_name, 'response': text})
        accumulated += f'\n[{spec_name.upper()}]: {text}\n'

    # Supervisor synthesis ONLY when 2+ specialists ran.
    if len(ordered) == 1:
        final = steps[0]['response']
        if verbose:
            print('\n(single specialist -> synthesis skipped)')
    else:
        if verbose:
            print('\n--- SUPERVISOR synthesizing ---')
        synth = (
            f'The engineer asked: "{user_input}"\n\n'
            f'Specialists responded in this order:\n{accumulated}\n\n'
            'Synthesize all of this into one clear response for the engineer. '
            'Present the facts, data, and any flags. Do not add recommendations '
            'beyond what was asked.'
        )
        final = str(supervisor(synth))
    return {'route': ordered, 'steps': steps, 'response': final}

result = run_graph('Which reaction wheels have at least 0.05 Nms of momentum storage?')
print('\n================ FINAL ANSWER ================\n')
print(result['response'])

1. CATALOG —

 find reaction wheels with momentum storage of at least 0.05 NmsROUTE: ['catalog']

--- CATALOG specialist running ---



(single specialist -> synthesis skipped)

================ FINAL ANSWER ================

Four of the 7 reaction wheels in the catalog have at least 0.05 Nms of momentum storage. The vendor for all four is AnyCompany. I got this from an exhaustive filter on `momentum_storage_nms` ≥ 0.05, so no matching wheel was missed.

| Part ID | Name | Momentum (Nms) | Max torque (mNm) | Mass (g) | Supply voltage (V) | Interface | Power (W) | Lifecycle |
|---|---|---|---|---|---|---|---|---|
| rw_rl_0060 | AnyCompany RW3 0.06 | 0.06 | 20 | 235 | 28 nominal (7.5–36) | RS-422 or redundant RS-485 | 1.3 | Active, full production |
| rw_rl_0400 | AnyCompany RW4 0.4 | 0.4 | 100 | 755 | 28 nominal (22–34) | RS-422 or redundant RS-485 | 2.8 | Active, full production |
| rw_rl_1000 | AnyCompany RW4 1.0 | 1.0 | 100 | 1400 | 28 nominal (22–34) | RS-422 or redundant RS-485 | 5 | Active, full production |
| rw_rl_5000 | AnyCompany RW 5.0 | 5.0 | 150 | 3000 | 28 nominal (22–34) | RS-422 or redundant RS-485 | 20

## 4 · Run a chained multi-specialist query (synthesis runs)

Now a query that genuinely needs a sequence. The planner routes
`knowledge → catalog → evaluation`, each specialist receives the accumulated
context from the ones before it, and because **more than one** specialist ran,
the Opus supervisor merges them into a single answer.

> This makes several Bedrock calls in sequence and can take 60–140 seconds.

In [6]:
result = run_graph(
    'Is the RW 0.03 at risk of going end-of-life? '
    'If so, find alternative reaction wheels and score them against a 0.03 Nms requirement.'
)
print('\n================ FINAL ANSWER ================\n')
print(result['response'])

1. KNOWLEDGE — check EOL status of the RW 0.03 via web search
2. CATALOG — find alternative reaction wheels matching a 0.03 Nms

 requirement
3. EVALUATION — score candidates against the 0.03 Nms requirementROUTE: ['knowledge', 'catalog', 'evaluation']

--- KNOWLEDGE specialist running ---


--- CATALOG specialist running ---


--- EVALUATION specialist running ---



--- SUPERVISOR synthesizing ---


...

 The RW4 0.4 is a large annular ring with a circular flange bolt pattern.
- **Interchangeability:** The photos suggest the two could not share a bracket. A sw

ap would likely need new mounting structure and harness work.
- **Limits:** There is no scale reference in the photos, and no electrical connector could be identified in either image.

---

This

 is evidence only, not a recommendation or appro

val of any substitution. The decision is yours.


================ FINAL ANSWER ================

# RW 0.03 End-of-Life Risk and Contingency Alternatives

**Routing:**
1. **KNOWLEDGE** handled the EOL and market status question.
2. **CATALOG** then searched for alternative wheels with at least 0.03 Nms.
3. **EVALUATION** scored those candidates spec by spec and ran one visual comparison.

## 1. EOL risk: none found

The RW 0.03 (part ID rw_rl_0030, vendor AnyCompany, model RW-0.03) is not at risk of end-of-life. The catalog and evaluation data agree:

- **Lifecycle:** Active, in full production
- **Estimated time to EOL:** 10+ years, with no last-time-buy announced
- **Lead time:** about 10 weeks, with low lead-time risk and stable demand
- **Heritage:** 90 units and 510+ cumulative years since 2008
- **Fit to your requirement:** 0.03 Nms momentum storage, 1.5 mNm maximum torque, 185 g mass

**Caveats:**
- **The web check used simulated demo results, not live data.** The vendor's product-line page reports no announced EOL. The lead-t

## 5 · The real orchestrator

What we hand-rolled in `run_graph()` above is exactly what `run_multi_agent()`
does in `agent/agents.py`. You can call it directly — this is the function the
non-streaming code path uses.

In [7]:
from mei_core.agents import run_multi_agent

out = run_multi_agent('What is the flight heritage of the RW3 0.06?')
print('routing_plan      :', out['routing_plan'].splitlines()[0])
print('specialists_used  :', out['specialists_used'])
print()
print(out['response'])

1. KNOWLEDGE — retrieve flight heritage of the RW3 0.06 reaction wheel

routing_plan      : 1. KNOWLEDGE — retrieve flight heritage of the RW3 0.06 reaction wheel
specialists_used  : ['knowledge']

The AnyCompany RW3-0.06 (catalog ID rw_rl_0060) has the following flight heritage in the catalog:

- **Units flown:** 317.
- **Time in orbit:** more than 930 cumulative years since 2014.
- **Standing:** the catalog calls it the vendor's most-flown model.
- **Qualification:** AnyCompany internal standard, described as "flight heritage qualified." It lists 14.1 grms vibration and 20 krad radiation tolerance.
- **Lifecycle:** Active and in full production, with 10+ years estimated to EOL. Lead time is about 10 weeks, rated low risk.

This comes from the catalog (get_part_specs) only. I didn't run a web search, and the catalog doesn't name specific missions.



## Recap

- **Planner (Haiku)** turns a free-form question into an ordered specialist plan.
- **Specialists (Sonnet)** are narrow and tool-focused; each one adds its piece,
  passing accumulated context forward.
- **Supervisor (Opus)** synthesizes — but only when 2+ specialists ran, so simple
  queries don't pay for a redundant model round trip.
- `run_multi_agent()` in `agent/agents.py` is the production entry point for the
  non-streaming path.

**Next:** [`02_specialist_tools_deep_dive.ipynb`](02_specialist_tools_deep_dive.ipynb)
opens up the deterministic `@tool` functions the specialists call — the evidence
layer underneath the reasoning.